In [15]:
!pip install --quiet  -r req-fine-models.txt

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [16]:
##Importante tener cuenta en hugging face, para algunos modelos como meta/llama 
# se debe tener aprobacion para descargar desde la plataforma el modelo.
from huggingface_hub import login
login(new_session=False)

In [17]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [18]:
import numpy as np
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
import transformers
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed,BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import random
from datasets import Dataset
from pathlib import Path
from sklearn.model_selection import train_test_split
from pathlib import Path
DATA_RAW = Path("./data-sources/raw")
DATA_PREPR = Path("./data-sources/pre-processed")
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PREPR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [19]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

In [20]:
df = pd.read_csv(DATA_PREPR/"data_finetuning.csv")
if 'name' in df.columns:
    df = df.drop(columns=['name'])

##Por la prueba

#df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)

# crea una máscara con el conteo de "tokens" por espacio
mask = df['article'].fillna('').astype(str).apply(lambda x: len(x.split())) <= 800

# opción B: reemplazar el DF original (in place)
df = df.loc[mask].reset_index(drop=True)

##Por la prueba

df_train, df_temp = train_test_split(df, test_size=0.3, random_state=42, shuffle=True)
df_test, df_val = train_test_split(df_temp, test_size=0.5, random_state=42, shuffle=True)

print(f'Columnas {df.columns}')
print(f'Registros totales cargados para entrenamiento: {len(df_train)}')
print(f'Registros totales cargados para validacion: {len(df_val)}')
print(f'Registros totales cargados para validacion: {len(df_test)}')

Columnas Index(['article', 'summary'], dtype='object')
Registros totales cargados para entrenamiento: 1726
Registros totales cargados para validacion: 370
Registros totales cargados para validacion: 370


In [21]:
def generar_prompt(cientific_text):
    input_text = f"""You are a helpful medical/health writer.
Convert the following scientific text into a clear, accurate Plain Language Summary (PLS) for a general audience.
Keep it concise, factual, and avoid jargon. If you must use a technical term, briefly define it.

Cientific Text: {cientific_text}

PLS Text:"""
    return input_text

In [22]:
def tokenize_with_labels(example):
    cientific_text = example['article']
    pls_text = example['summary']

    input_text = generar_prompt(cientific_text)

    output_text = f"{pls_text}"+tokenizer.eos_token

    input_tokens = tokenizer(input_text, add_special_tokens=False)
    output_tokens = tokenizer(output_text, add_special_tokens=False)

    input_ids = input_tokens["input_ids"] + output_tokens["input_ids"]
    labels = [-100] * len(input_tokens["input_ids"]) + output_tokens["input_ids"]

    return {
        "input_ids": input_ids,
        "labels": labels
    }

def simple_collate(batch):
    input_ids = [torch.tensor(example["input_ids"], dtype=torch.long) for example in batch]
    labels = [torch.tensor(example["labels"], dtype=torch.long) for example in batch]

    input_ids_padded = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=-100)

    attention_mask = (input_ids_padded != tokenizer.pad_token_id).long()

    return {
        "input_ids": input_ids_padded,
        "attention_mask": attention_mask,
        "labels": labels_padded
    }


In [ ]:
##Si la intencion es cargar el modelo con los pesos pre entrenados
#  ignorar las siguientes celdas

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

In [ ]:
lora_config = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=['k_proj', 'q_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
quant_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16))
tunned_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B", dtype=torch.float16,
                                                        quantization_config=quant_4bit, device_map='auto')

tunned_model = get_peft_model(tunned_model, lora_config)
print("modelo:", getattr(tunned_model, "name_or_path", type(tunned_model)))
print("is_loaded_in_4bit:", getattr(tunned_model, "is_loaded_in_4bit", False))
print("is_loaded_in_8bit:", getattr(tunned_model, "is_loaded_in_8bit", False))
print("dtype params:", next(tunned_model.parameters()).dtype)
print("PEFT activo:", hasattr(tunned_model, "peft_config"))

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

modelo: meta-llama/Llama-3.2-1B
is_loaded_in_4bit: True
is_loaded_in_8bit: False
dtype params: torch.float16
PEFT activo: True


In [ ]:
train_data = Dataset.from_pandas(df_train)
val_data = Dataset.from_pandas(df_val)

train_data = train_data.map(tokenize_with_labels)
val_data = val_data.map(tokenize_with_labels)

Map:   0%|          | 0/1726 [00:00<?, ? examples/s]

Map:   0%|          | 0/370 [00:00<?, ? examples/s]

In [ ]:
import math

BATCH_SIZE = 1
ACCUM_GRAD_STEPS = 16
EPOCHS = 1

MAX_STEPS =  int(EPOCHS * math.ceil(train_data.num_rows / (BATCH_SIZE * ACCUM_GRAD_STEPS)))
print(MAX_STEPS)
WARMUP = int(MAX_STEPS * 0.1)
print(WARMUP)
EVAL_STEPS = int(MAX_STEPS / 4)
print(EVAL_STEPS)

108
10
27


In [ ]:
training_args =transformers.TrainingArguments(
    data_seed = RANDOM_SEED,
    seed = RANDOM_SEED,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=ACCUM_GRAD_STEPS,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy='epoch',
    num_train_epochs=EPOCHS,
    lr_scheduler_type="cosine",
    warmup_steps= WARMUP,
    learning_rate=0.0003,
    weight_decay=0.01,
    fp16=True,
    output_dir='outputs',
    save_strategy="no",
    report_to="none"
)


trainer = transformers.Trainer(
    model=tunned_model,
    train_dataset=train_data,
    eval_dataset=val_data,
    args=training_args,
    data_collator=simple_collate
)
trainer.train()

trainer.save_model("models/llama3.2-1b")
tokenizer.save_pretrained("models/llama3.2-1b")

Epoch,Training Loss,Validation Loss
1,No log,1.544544


('outputs/llama-pls/tokenizer_config.json',
 'outputs/llama-pls/special_tokens_map.json',
 'outputs/llama-pls/tokenizer.json')

In [10]:
##Continuar desde aca para cargar el modelo con los pesos luego del finetuning

In [23]:
adapters_path = "models/llama3.2-1b"

tokenizer = AutoTokenizer.from_pretrained(adapters_path, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

#no olvidar colocar device_map='auto' al estar en gpu
model = AutoModelForCausalLM.from_pretrained(adapters_path,dtype=torch.float16).eval()
model.config.use_cache = True

In [24]:

@torch.inference_mode()
def generate_pls_llama(article: str,
                       max_new_tokens: int = 1024,
                       temperature: float = 0.01,
                       top_p: float = 0.9,
                       num_beams: int = 1) -> str:
    prompt = generar_prompt(article)

    max_ctx = getattr(model.config, "max_position_embeddings", 4096)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_ctx- max_new_tokens)
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    gen_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=(num_beams == 1),     
        temperature=temperature,
        top_p=top_p,
        num_beams=num_beams,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True
    )

    gen_only = gen_ids[0, input_ids.shape[1]:]
    text = tokenizer.decode(gen_only, skip_special_tokens=True)
    return text.strip()


In [25]:
## Ejemplo de 1 prediccion
print("** Articulo cientifico: ",df_test.iloc[2]['article'])
print("** Articulo resumido: ",df_test.iloc[2]['summary'])
print("** Resumen generado: ",generate_pls_llama(df_test.iloc[2]['article']))

** Articulo cientifico:  Background
Osteoarthritis is the most common form of joint disorder and a leading cause of pain and physical disability. Observational studies suggested a benefit for joint lavage, but recent, sham‐controlled trials yielded conflicting results, suggesting joint lavage not to be effective. 
Objectives
To compare joint lavage with sham intervention, placebo or non‐intervention control in terms of effects on pain, function and safety outcomes in patients with knee osteoarthritis. 
Search methods
We searched CENTRAL, MEDLINE, EMBASE, and CINAHL up to 3 August 2009, checked conference proceedings, reference lists, and contacted authors. 
Selection criteria
We included studies if they were randomised or quasi‐randomised trials that compared arthroscopic and non‐arthroscopic joint lavage with a control intervention in patients with osteoarthritis of the knee. We did not apply any language restrictions. 
Data collection and analysis
Two independent review authors extra

In [ ]:
#TODO: Espacio para agregar el metodo que tomando todos los textos cientificos del dataset de test 
# genere los resumenes y los guarde en un csv para que posteriormente se haga el calculo de todas las metricas.
